In [1]:
# ================================================================
# 03_ro3_temporal_robustness.py
#
# RO3: Temporal robustness evaluation
#
# Training:
#     S_old_temporal_train.csv
#
# Testing:
#     S_new_temporal_test.csv
#
# Important:
#     S_new is NEVER used for tuning.
# ================================================================

import os
import time
import warnings

import joblib
import numpy as np
import pandas as pd
import optuna

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor


warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)



In [ ]:
# ================================================================
# 1. SETTINGS
# ================================================================

RANDOM_STATE = 42
N_TRIALS = 3
N_SPLITS = 5

S_OLD_PATH = "../data/processed/S_old_temporal_train.csv"
S_NEW_PATH = "../data/processed/S_new_temporal_test.csv"

MODEL_DIR = "../models"
TABLE_DIR = "../results/tables"

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(TABLE_DIR, exist_ok=True)



In [3]:
# ================================================================
# 2. LOAD DATA
# ================================================================

old_df = pd.read_csv(S_OLD_PATH)
new_df = pd.read_csv(S_NEW_PATH)

print("=" * 70)
print("RO3 TEMPORAL ROBUSTNESS EVALUATION")
print("=" * 70)

print("\nS_old shape:", old_df.shape)
print("S_new shape:", new_df.shape)



RO3 TEMPORAL ROBUSTNESS EVALUATION

S_old shape: (1983, 15)
S_new shape: (4002, 15)


In [ ]:
# ================================================================
# 3. FEATURES
# ================================================================

categorical_features = [
    "make",
    "model",
    "vehicle_class",
    "transmission",
    "fuel_type"
]

numeric_features = [
    "engine_size",
    "cylinders",
    "transmission_gears",
    "engine_per_cylinder",
    "engine_x_cylinders",
    "engine_size_sq",
    "cylinders_sq"
]

target = "co2"

feature_columns = numeric_features + categorical_features


# Check required columns

missing_old = [
    col for col in feature_columns + [target]
    if col not in old_df.columns
]

missing_new = [
    col for col in feature_columns + [target]
    if col not in new_df.columns
]

if missing_old:
    raise ValueError(
        f"Missing columns in S_old: {missing_old}"
    )

if missing_new:
    raise ValueError(
        f"Missing columns in S_new: {missing_new}"
    )



In [5]:
# ================================================================
# 4. X AND y
# ================================================================

X_old = old_df[feature_columns].copy()
y_old = old_df[target].copy()

X_new = new_df[feature_columns].copy()
y_new = new_df[target].copy()

print("\nTraining observations:", len(X_old))
print("Temporal test observations:", len(X_new))

print("\nFeatures:")
print(feature_columns)




Training observations: 1983
Temporal test observations: 4002

Features:
['engine_size', 'cylinders', 'engine_per_cylinder', 'engine_x_cylinders', 'engine_size_sq', 'cylinders_sq', 'make', 'model', 'vehicle_class', 'transmission', 'fuel_type']


In [ ]:
# ================================================================
# 5. PREPROCESSING
# ================================================================

preprocess = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_features
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)



In [7]:
# ================================================================
# 6. CROSS-VALIDATION
# ================================================================

kf = KFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)



In [8]:
# ================================================================
# 7. METRIC FUNCTION
# ================================================================

def calculate_metrics(y_true, y_pred):

    return {
        "R2": r2_score(y_true, y_pred),
        "RMSE": np.sqrt(
            mean_squared_error(y_true, y_pred)
        ),
        "MAE": mean_absolute_error(
            y_true,
            y_pred
        ),
        "N": len(y_true)
    }



In [9]:
# ================================================================
# 8. SUBSET METRIC FUNCTION
# ================================================================

def calculate_subset_metrics(
    y_true,
    y_pred,
    mask
):

    if mask.sum() == 0:

        return {
            "R2": np.nan,
            "RMSE": np.nan,
            "MAE": np.nan,
            "N": 0
        }

    y_true_subset = np.asarray(y_true)[mask]
    y_pred_subset = np.asarray(y_pred)[mask]

    return calculate_metrics(
        y_true_subset,
        y_pred_subset
    )



In [10]:
# ================================================================
# 9. OPTUNA FUNCTION
# ================================================================

def run_optuna(objective):

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(
            seed=RANDOM_STATE
        )
    )

    study.optimize(
        objective,
        n_trials=N_TRIALS,
        show_progress_bar=False
    )

    return study



In [11]:
# ================================================================
# 10. RANDOM FOREST
# ================================================================

def tune_random_forest():

    def objective(trial):

        params = {
            "n_estimators": trial.suggest_int(
                "n_estimators",
                100,
                600
            ),

            "max_depth": trial.suggest_int(
                "max_depth",
                5,
                20
            ),

            "min_samples_split": trial.suggest_int(
                "min_samples_split",
                2,
                15
            ),

            "min_samples_leaf": trial.suggest_int(
                "min_samples_leaf",
                1,
                10
            ),

            "max_features": trial.suggest_categorical(
                "max_features",
                ["sqrt", "log2"]
            ),

            "random_state": RANDOM_STATE,
            "n_jobs": -1
        }

        model = Pipeline(
            [
                ("preprocess", preprocess),

                (
                    "model",
                    RandomForestRegressor(**params)
                )
            ]
        )

        scores = cross_val_score(
            model,
            X_old,
            y_old,
            cv=kf,
            scoring="r2",
            n_jobs=-1
        )

        return scores.mean()

    study = run_optuna(objective)

    best_params = study.best_params

    best_params["random_state"] = RANDOM_STATE
    best_params["n_jobs"] = -1

    model = Pipeline(
        [
            ("preprocess", preprocess),

            (
                "model",
                RandomForestRegressor(**best_params)
            )
        ]
    )

    return study, model



In [12]:
# ================================================================
# 11. XGBOOST
# ================================================================

def tune_xgboost():

    def objective(trial):

        params = {
            "n_estimators": trial.suggest_int(
                "n_estimators",
                100,
                500
            ),

            "max_depth": trial.suggest_int(
                "max_depth",
                3,
                10
            ),

            "learning_rate": trial.suggest_float(
                "learning_rate",
                0.01,
                0.2,
                log=True
            ),

            "subsample": trial.suggest_float(
                "subsample",
                0.6,
                1.0
            ),

            "colsample_bytree": trial.suggest_float(
                "colsample_bytree",
                0.6,
                1.0
            ),

            "gamma": trial.suggest_float(
                "gamma",
                0,
                5
            ),

            "objective": "reg:squarederror",

            "random_state": RANDOM_STATE,

            "n_jobs": -1
        }

        model = Pipeline(
            [
                ("preprocess", preprocess),

                (
                    "model",
                    XGBRegressor(**params)
                )
            ]
        )

        scores = cross_val_score(
            model,
            X_old,
            y_old,
            cv=kf,
            scoring="r2",
            n_jobs=-1
        )

        return scores.mean()

    study = run_optuna(objective)

    best_params = study.best_params

    best_params["objective"] = "reg:squarederror"
    best_params["random_state"] = RANDOM_STATE
    best_params["n_jobs"] = -1

    model = Pipeline(
        [
            ("preprocess", preprocess),

            (
                "model",
                XGBRegressor(**best_params)
            )
        ]
    )

    return study, model



In [13]:
# ================================================================
# 12. RIDGE
# ================================================================

def tune_ridge():

    def objective(trial):

        alpha = trial.suggest_float(
            "alpha",
            1e-4,
            1e2,
            log=True
        )

        model = Pipeline(
            [
                ("preprocess", preprocess),

                (
                    "model",
                    Ridge(alpha=alpha)
                )
            ]
        )

        scores = cross_val_score(
            model,
            X_old,
            y_old,
            cv=kf,
            scoring="r2",
            n_jobs=-1
        )

        return scores.mean()

    study = run_optuna(objective)

    model = Pipeline(
        [
            ("preprocess", preprocess),

            (
                "model",
                Ridge(
                    alpha=study.best_params["alpha"]
                )
            )
        ]
    )

    return study, model



In [14]:
# ================================================================
# 13. LIGHTGBM WITH ONE-HOT ENCODING
# ================================================================

def tune_lightgbm():

    def objective(trial):

        params = {
            "n_estimators": trial.suggest_int(
                "n_estimators",
                100,
                500
            ),

            "learning_rate": trial.suggest_float(
                "learning_rate",
                0.01,
                0.2,
                log=True
            ),

            "max_depth": trial.suggest_int(
                "max_depth",
                3,
                10
            ),

            "num_leaves": trial.suggest_int(
                "num_leaves",
                15,
                63
            ),

            "random_state": RANDOM_STATE,

            "n_jobs": -1,

            "verbosity": -1
        }

        model = Pipeline(
            [
                ("preprocess", preprocess),

                (
                    "model",
                    LGBMRegressor(**params)
                )
            ]
        )

        scores = cross_val_score(
            model,
            X_old,
            y_old,
            cv=kf,
            scoring="r2",
            n_jobs=-1
        )

        return scores.mean()

    study = run_optuna(objective)

    best_params = study.best_params

    best_params["random_state"] = RANDOM_STATE
    best_params["n_jobs"] = -1
    best_params["verbosity"] = -1

    model = Pipeline(
        [
            ("preprocess", preprocess),

            (
                "model",
                LGBMRegressor(**best_params)
            )
        ]
    )

    return study, model



In [15]:
# ================================================================
# 14. CATBOOST WITH ONE-HOT ENCODING
# ================================================================

def tune_catboost():

    def objective(trial):

        params = {
            "iterations": trial.suggest_int(
                "iterations",
                100,
                500
            ),

            "depth": trial.suggest_int(
                "depth",
                4,
                10
            ),

            "learning_rate": trial.suggest_float(
                "learning_rate",
                0.01,
                0.2,
                log=True
            ),

            "l2_leaf_reg": trial.suggest_float(
                "l2_leaf_reg",
                1,
                10
            ),

            "loss_function": "RMSE",

            "random_seed": RANDOM_STATE,

            "verbose": False,

            "thread_count": -1
        }

        model = Pipeline(
            [
                ("preprocess", preprocess),

                (
                    "model",
                    CatBoostRegressor(**params)
                )
            ]
        )

        scores = cross_val_score(
            model,
            X_old,
            y_old,
            cv=kf,
            scoring="r2",
            n_jobs=1
        )

        return scores.mean()

    study = run_optuna(objective)

    best_params = study.best_params

    best_params["loss_function"] = "RMSE"
    best_params["random_seed"] = RANDOM_STATE
    best_params["verbose"] = False
    best_params["thread_count"] = -1

    model = Pipeline(
        [
            ("preprocess", preprocess),

            (
                "model",
                CatBoostRegressor(**best_params)
            )
        ]
    )

    return study, model



In [ ]:
# ================================================================
# 15. PREPARE NATIVE CATEGORICAL DATA
# ================================================================

def prepare_native_data(
    X_train,
    X_test
):

    X_train = X_train.copy()
    X_test = X_test.copy()

    for col in categorical_features:

        categories = (
            X_train[col]
            .astype(str)
            .unique()
        )

        X_train[col] = pd.Categorical(
            X_train[col].astype(str),
            categories=categories
        )

        X_test[col] = pd.Categorical(
            X_test[col].astype(str),
            categories=categories
        )

    return X_train, X_test



In [17]:
# ================================================================
# 16. LIGHTGBM NATIVE CATEGORICAL
# ================================================================

def tune_lightgbm_native():

    X_train_native, _ = prepare_native_data(
        X_old,
        X_old
    )

    def objective(trial):

        params = {
            "n_estimators": trial.suggest_int(
                "n_estimators",
                100,
                500
            ),

            "learning_rate": trial.suggest_float(
                "learning_rate",
                0.01,
                0.2,
                log=True
            ),

            "max_depth": trial.suggest_int(
                "max_depth",
                3,
                10
            ),

            "num_leaves": trial.suggest_int(
                "num_leaves",
                15,
                63
            ),

            "random_state": RANDOM_STATE,

            "n_jobs": -1,

            "verbosity": -1
        }

        fold_scores = []

        for train_idx, valid_idx in kf.split(
            X_train_native
        ):

            X_train_fold = (
                X_train_native.iloc[train_idx]
            )

            X_valid_fold = (
                X_train_native.iloc[valid_idx]
            )

            y_train_fold = y_old.iloc[train_idx]
            y_valid_fold = y_old.iloc[valid_idx]

            model = LGBMRegressor(**params)

            model.fit(
                X_train_fold,
                y_train_fold,
                categorical_feature=categorical_features
            )

            prediction = model.predict(
                X_valid_fold
            )

            score = r2_score(
                y_valid_fold,
                prediction
            )

            fold_scores.append(score)

        return np.mean(fold_scores)

    study = run_optuna(objective)

    best_params = study.best_params

    best_params["random_state"] = RANDOM_STATE
    best_params["n_jobs"] = -1
    best_params["verbosity"] = -1

    return study, best_params



In [18]:
# ================================================================
# 17. CATBOOST NATIVE CATEGORICAL
# ================================================================

def tune_catboost_native():

    X_native = X_old.copy()

    for col in categorical_features:
        X_native[col] = X_native[col].astype(str)

    categorical_indices = [
        X_native.columns.get_loc(col)
        for col in categorical_features
    ]

    def objective(trial):

        params = {
            "iterations": trial.suggest_int(
                "iterations",
                100,
                500
            ),

            "depth": trial.suggest_int(
                "depth",
                4,
                10
            ),

            "learning_rate": trial.suggest_float(
                "learning_rate",
                0.01,
                0.2,
                log=True
            ),

            "l2_leaf_reg": trial.suggest_float(
                "l2_leaf_reg",
                1,
                10
            ),

            "loss_function": "RMSE",

            "random_seed": RANDOM_STATE,

            "verbose": False,

            "thread_count": -1
        }

        fold_scores = []

        for train_idx, valid_idx in kf.split(
            X_native
        ):

            X_train_fold = X_native.iloc[train_idx]
            X_valid_fold = X_native.iloc[valid_idx]

            y_train_fold = y_old.iloc[train_idx]
            y_valid_fold = y_old.iloc[valid_idx]

            model = CatBoostRegressor(
                **params
            )

            model.fit(
                X_train_fold,
                y_train_fold,
                cat_features=categorical_indices,
                verbose=False
            )

            prediction = model.predict(
                X_valid_fold
            )

            score = r2_score(
                y_valid_fold,
                prediction
            )

            fold_scores.append(score)

        return np.mean(fold_scores)

    study = run_optuna(objective)

    best_params = study.best_params

    best_params["loss_function"] = "RMSE"
    best_params["random_seed"] = RANDOM_STATE
    best_params["verbose"] = False
    best_params["thread_count"] = -1

    return study, best_params



In [19]:
# ================================================================
# 18. EVALUATE A MODEL
# ================================================================

def evaluate_model(
    model_name,
    model,
    X_train,
    y_train,
    X_test,
    y_test,
    ood_mask,
    extreme_mask,
    model_filename
):

    print("\n" + "=" * 70)
    print(model_name)
    print("=" * 70)

    # ------------------------------------------------------------
    # Train
    # ------------------------------------------------------------

    start_time = time.time()

    model.fit(
        X_train,
        y_train
    )

    training_time = (
        time.time() - start_time
    )

    # ------------------------------------------------------------
    # Predict
    # ------------------------------------------------------------

    start_time = time.time()

    prediction = model.predict(
        X_test
    )

    prediction_time = (
        time.time() - start_time
    )

    # ------------------------------------------------------------
    # Full test
    # ------------------------------------------------------------

    full_metrics = calculate_metrics(
        y_test,
        prediction
    )

    # ------------------------------------------------------------
    # OOD
    # ------------------------------------------------------------

    ood_metrics = calculate_subset_metrics(
        y_test,
        prediction,
        ood_mask
    )

    # ------------------------------------------------------------
    # Extreme tail
    # ------------------------------------------------------------

    extreme_metrics = calculate_subset_metrics(
        y_test,
        prediction,
        extreme_mask
    )

    # ------------------------------------------------------------
    # Print
    # ------------------------------------------------------------

    print(
        "\nFull temporal test:"
    )

    print(
        f"R2   : {full_metrics['R2']:.4f}"
    )

    print(
        f"RMSE : {full_metrics['RMSE']:.4f}"
    )

    print(
        f"MAE  : {full_metrics['MAE']:.4f}"
    )

    print(
        f"N    : {full_metrics['N']}"
    )

    print(
        "\nStrict OOD:"
    )

    print(
        f"R2   : {ood_metrics['R2']:.4f}"
    )

    print(
        f"RMSE : {ood_metrics['RMSE']:.4f}"
    )

    print(
        f"MAE  : {ood_metrics['MAE']:.4f}"
    )

    print(
        f"N    : {ood_metrics['N']}"
    )

    print(
        "\nExtreme-tail stress test:"
    )

    print(
        f"R2   : {extreme_metrics['R2']:.4f}"
    )

    print(
        f"RMSE : {extreme_metrics['RMSE']:.4f}"
    )

    print(
        f"MAE  : {extreme_metrics['MAE']:.4f}"
    )

    print(
        f"N    : {extreme_metrics['N']}"
    )

    print(
        f"\nTraining time: {training_time:.2f} seconds"
    )

    print(
        f"Prediction time: {prediction_time:.2f} seconds"
    )

    # ------------------------------------------------------------
    # Save model
    # ------------------------------------------------------------

    model_path = os.path.join(
        MODEL_DIR,
        model_filename
    )

    joblib.dump(
        model,
        model_path
    )

    print(
        "\nSaved model:",
        model_path
    )

    # ------------------------------------------------------------
    # Save predictions
    # ------------------------------------------------------------

    prediction_df = new_df.copy()

    prediction_df["prediction"] = prediction

    prediction_df["residual"] = (
        y_test.to_numpy()
        - prediction
    )

    prediction_df["is_ood"] = ood_mask

    prediction_df["is_extreme_tail"] = (
        extreme_mask
    )

    prediction_filename = (
        model_filename.replace(
            ".joblib",
            "_predictions.csv"
        )
    )

    prediction_path = os.path.join(
        TABLE_DIR,
        prediction_filename
    )

    prediction_df.to_csv(
        prediction_path,
        index=False
    )

    print(
        "Saved predictions:",
        prediction_path
    )

    # ------------------------------------------------------------
    # Results
    # ------------------------------------------------------------

    rows = [

        {
            "model": model_name,
            "subset": "Full temporal test",
            "R2": full_metrics["R2"],
            "RMSE": full_metrics["RMSE"],
            "MAE": full_metrics["MAE"],
            "N": full_metrics["N"],
            "training_time_s": training_time,
            "prediction_time_s": prediction_time
        },

        {
            "model": model_name,
            "subset": "Strict OOD",
            "R2": ood_metrics["R2"],
            "RMSE": ood_metrics["RMSE"],
            "MAE": ood_metrics["MAE"],
            "N": ood_metrics["N"],
            "training_time_s": np.nan,
            "prediction_time_s": np.nan
        },

        {
            "model": model_name,
            "subset": "Extreme-tail stress test",
            "R2": extreme_metrics["R2"],
            "RMSE": extreme_metrics["RMSE"],
            "MAE": extreme_metrics["MAE"],
            "N": extreme_metrics["N"],
            "training_time_s": np.nan,
            "prediction_time_s": np.nan
        }
    ]

    return prediction, rows



In [20]:
# ================================================================
# 19. DEFINE OOD THRESHOLDS
#
# IMPORTANT:
# These thresholds are calculated ONLY from S_old.
# ================================================================

engine_min = X_old["engine_size"].min()
engine_max = X_old["engine_size"].max()

engine_p05 = X_old["engine_size"].quantile(
    0.05
)

engine_p95 = X_old["engine_size"].quantile(
    0.95
)

print("\n" + "=" * 70)
print("ROBUSTNESS THRESHOLDS")
print("=" * 70)

print(
    "S_old engine_size minimum:",
    engine_min
)

print(
    "S_old engine_size maximum:",
    engine_max
)

print(
    "S_old engine_size 5th percentile:",
    engine_p05
)

print(
    "S_old engine_size 95th percentile:",
    engine_p95
)




ROBUSTNESS THRESHOLDS
S_old engine_size minimum: 2.5
S_old engine_size maximum: 8.4
S_old engine_size 5th percentile: 3.0
S_old engine_size 95th percentile: 6.4


In [21]:
# ================================================================
# 20. STRICT OOD MASK
#
# A test observation is OOD if its engine size is outside
# the minimum/maximum engine-size range observed in S_old.
# ================================================================

ood_mask = (
    (X_new["engine_size"] < engine_min)
    |
    (X_new["engine_size"] > engine_max)
).to_numpy()



In [22]:
# ================================================================
# 21. EXTREME-TAIL MASK
#
# This is NOT called OOD.
#
# It identifies observations in the lower/upper 5% tails
# according to thresholds calculated from S_old.
# ================================================================

extreme_mask = (
    (X_new["engine_size"] <= engine_p05)
    |
    (X_new["engine_size"] >= engine_p95)
).to_numpy()


print(
    "\nS_new observations:",
    len(X_new)
)

print(
    "Strict OOD observations:",
    int(ood_mask.sum())
)

print(
    "Extreme-tail observations:",
    int(extreme_mask.sum())
)




S_new observations: 4002
Strict OOD observations: 2346
Extreme-tail observations: 3343


In [23]:
# ================================================================
# 22. STORAGE FOR RESULTS
# ================================================================

all_results = []

all_predictions = pd.DataFrame(
    {
        "make": new_df["make"],
        "model": new_df["model"],
        "vehicle_class": new_df["vehicle_class"],
        "engine_size": new_df["engine_size"],
        "cylinders": new_df["cylinders"],
        "co2": y_new,
        "is_ood": ood_mask,
        "is_extreme_tail": extreme_mask
    }
)



In [24]:
# ================================================================
# 23. RANDOM FOREST
# ================================================================

print("\n" + "=" * 70)
print("TUNING RANDOM FOREST")
print("=" * 70)

rf_study, rf_model = tune_random_forest()

print(
    "Best CV R2:",
    rf_study.best_value
)

print(
    "Best parameters:",
    rf_study.best_params
)

rf_prediction, rf_results = evaluate_model(
    "Random Forest (Tuned)",
    rf_model,
    X_old,
    y_old,
    X_new,
    y_new,
    ood_mask,
    extreme_mask,
    "ro3_random_forest_temporal.joblib"
)

all_results.extend(
    rf_results
)

all_predictions[
    "Random Forest (Tuned)"
] = rf_prediction




TUNING RANDOM FOREST
Best CV R2: 0.8491738363664197
Best parameters: {'n_estimators': 239, 'max_depth': 20, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt'}

Random Forest (Tuned)

Full temporal test:
R2   : -0.9026
RMSE : 56.7739
MAE  : 48.1060
N    : 4002

Strict OOD:
R2   : -4.4126
RMSE : 66.9184
MAE  : 61.3057
N    : 2346

Extreme-tail stress test:
R2   : -1.7893
RMSE : 60.7424
MAE  : 52.9742
N    : 3343

Training time: 1.07 seconds
Prediction time: 0.15 seconds

Saved model: ../models\ro3_random_forest_temporal.joblib
Saved predictions: ../results/tables\ro3_random_forest_temporal_predictions.csv


In [25]:
# ================================================================
# 24. XGBOOST
# ================================================================

print("\n" + "=" * 70)
print("TUNING XGBOOST")
print("=" * 70)

xgb_study, xgb_model = tune_xgboost()

print(
    "Best CV R2:",
    xgb_study.best_value
)

print(
    "Best parameters:",
    xgb_study.best_params
)

xgb_prediction, xgb_results = evaluate_model(
    "XGBoost (Tuned)",
    xgb_model,
    X_old,
    y_old,
    X_new,
    y_new,
    ood_mask,
    extreme_mask,
    "ro3_xgb_temporal.joblib"
)

all_results.extend(
    xgb_results
)

all_predictions[
    "XGBoost (Tuned)"
] = xgb_prediction




TUNING XGBOOST
Best CV R2: 0.9319204211235046
Best parameters: {'n_estimators': 436, 'max_depth': 9, 'learning_rate': 0.1867333050133828, 'subsample': 0.7381510571732653, 'colsample_bytree': 0.7727745684770461, 'gamma': 4.166597187635957}

XGBoost (Tuned)

Full temporal test:
R2   : -0.3996
RMSE : 48.6937
MAE  : 39.4245
N    : 4002

Strict OOD:
R2   : -3.0029
RMSE : 57.5475
MAE  : 50.3546
N    : 2346

Extreme-tail stress test:
R2   : -1.0545
RMSE : 52.1306
MAE  : 43.4919
N    : 3343

Training time: 1.24 seconds
Prediction time: 0.06 seconds

Saved model: ../models\ro3_xgb_temporal.joblib
Saved predictions: ../results/tables\ro3_xgb_temporal_predictions.csv


In [26]:
# ================================================================
# 25. RIDGE
# ================================================================

print("\n" + "=" * 70)
print("TUNING RIDGE")
print("=" * 70)

ridge_study, ridge_model = tune_ridge()

print(
    "Best CV R2:",
    ridge_study.best_value
)

print(
    "Best parameters:",
    ridge_study.best_params
)

ridge_prediction, ridge_results = evaluate_model(
    "Ridge Regression (Tuned)",
    ridge_model,
    X_old,
    y_old,
    X_new,
    y_new,
    ood_mask,
    extreme_mask,
    "ro3_ridge_temporal.joblib"
)

all_results.extend(
    ridge_results
)

all_predictions[
    "Ridge Regression (Tuned)"
] = ridge_prediction




TUNING RIDGE
Best CV R2: 0.897675408534462
Best parameters: {'alpha': 0.2916841654276852}

Ridge Regression (Tuned)

Full temporal test:
R2   : 0.4539
RMSE : 30.4155
MAE  : 23.5185
N    : 4002

Strict OOD:
R2   : -0.1739
RMSE : 31.1641
MAE  : 25.1477
N    : 2346

Extreme-tail stress test:
R2   : 0.2466
RMSE : 31.5691
MAE  : 24.8258
N    : 3343

Training time: 0.04 seconds
Prediction time: 0.02 seconds

Saved model: ../models\ro3_ridge_temporal.joblib
Saved predictions: ../results/tables\ro3_ridge_temporal_predictions.csv


In [27]:
# ================================================================
# 26. LIGHTGBM
# ================================================================

print("\n" + "=" * 70)
print("TUNING LIGHTGBM")
print("=" * 70)

lgbm_study, lgbm_model = tune_lightgbm()

print(
    "Best CV R2:",
    lgbm_study.best_value
)

print(
    "Best parameters:",
    lgbm_study.best_params
)

lgbm_prediction, lgbm_results = evaluate_model(
    "LightGBM (Tuned)",
    lgbm_model,
    X_old,
    y_old,
    X_new,
    y_new,
    ood_mask,
    extreme_mask,
    "ro3_lightgbm_temporal.joblib"
)

all_results.extend(
    lgbm_results
)

all_predictions[
    "LightGBM (Tuned)"
] = lgbm_prediction




TUNING LIGHTGBM
Best CV R2: 0.8932962452044535
Best parameters: {'n_estimators': 482, 'learning_rate': 0.19727532129329317, 'max_depth': 8, 'num_leaves': 15}

LightGBM (Tuned)

Full temporal test:
R2   : -0.4363
RMSE : 49.3282
MAE  : 40.0207
N    : 4002

Strict OOD:
R2   : -3.1830
RMSE : 58.8281
MAE  : 51.3964
N    : 2346

Extreme-tail stress test:
R2   : -1.1093
RMSE : 52.8214
MAE  : 44.2198
N    : 3343

Training time: 0.39 seconds
Prediction time: 0.07 seconds

Saved model: ../models\ro3_lightgbm_temporal.joblib
Saved predictions: ../results/tables\ro3_lightgbm_temporal_predictions.csv


In [28]:
# ================================================================
# 27. CATBOOST
# ================================================================

print("\n" + "=" * 70)
print("TUNING CATBOOST")
print("=" * 70)

cat_study, cat_model = tune_catboost()

print(
    "Best CV R2:",
    cat_study.best_value
)

print(
    "Best parameters:",
    cat_study.best_params
)

cat_prediction, cat_results = evaluate_model(
    "CatBoost (Tuned)",
    cat_model,
    X_old,
    y_old,
    X_new,
    y_new,
    ood_mask,
    extreme_mask,
    "ro3_catboost_temporal.joblib"
)

all_results.extend(
    cat_results
)

all_predictions[
    "CatBoost (Tuned)"
] = cat_prediction




TUNING CATBOOST
Best CV R2: 0.9288320161275735
Best parameters: {'iterations': 491, 'depth': 9, 'learning_rate': 0.1595210737239484, 'l2_leaf_reg': 1.20964430670226}

CatBoost (Tuned)

Full temporal test:
R2   : -0.3688
RMSE : 48.1560
MAE  : 39.0699
N    : 4002

Strict OOD:
R2   : -2.8145
RMSE : 56.1776
MAE  : 49.0885
N    : 2346

Extreme-tail stress test:
R2   : -1.0144
RMSE : 51.6200
MAE  : 43.2217
N    : 3343

Training time: 27.28 seconds
Prediction time: 0.05 seconds

Saved model: ../models\ro3_catboost_temporal.joblib
Saved predictions: ../results/tables\ro3_catboost_temporal_predictions.csv


In [29]:
# ================================================================
# 28. LIGHTGBM NATIVE CATEGORICAL
# ================================================================

print("\n" + "=" * 70)
print("TUNING LIGHTGBM NATIVE CATEGORICAL")
print("=" * 70)

lgbm_native_study, lgbm_native_params = (
    tune_lightgbm_native()
)

print(
    "Best CV R2:",
    lgbm_native_study.best_value
)

print(
    "Best parameters:",
    lgbm_native_params
)


X_old_lgbm, X_new_lgbm = prepare_native_data(
    X_old,
    X_new
)

lgbm_native_model = LGBMRegressor(
    **lgbm_native_params
)


lgbm_native_prediction, lgbm_native_results = (
    evaluate_model(
        "LightGBM (Native categorical)",
        lgbm_native_model,
        X_old_lgbm,
        y_old,
        X_new_lgbm,
        y_new,
        ood_mask,
        extreme_mask,
        "ro3_lightgbm_native_temporal.joblib"
    )
)

all_results.extend(
    lgbm_native_results
)

all_predictions[
    "LightGBM (Native categorical)"
] = lgbm_native_prediction




TUNING LIGHTGBM NATIVE CATEGORICAL
Best CV R2: 0.8963752099802302
Best parameters: {'n_estimators': 400, 'learning_rate': 0.17721623473946552, 'max_depth': 5, 'num_leaves': 62, 'random_state': 42, 'n_jobs': -1, 'verbosity': -1}

LightGBM (Native categorical)

Full temporal test:
R2   : -0.3501
RMSE : 47.8261
MAE  : 39.3252
N    : 4002

Strict OOD:
R2   : -2.9472
RMSE : 57.1458
MAE  : 50.7860
N    : 2346

Extreme-tail stress test:
R2   : -0.9784
RMSE : 51.1565
MAE  : 43.4154
N    : 3343

Training time: 0.39 seconds
Prediction time: 0.04 seconds

Saved model: ../models\ro3_lightgbm_native_temporal.joblib
Saved predictions: ../results/tables\ro3_lightgbm_native_temporal_predictions.csv


In [30]:
# ================================================================
# 29. CATBOOST NATIVE CATEGORICAL
# ================================================================

print("\n" + "=" * 70)
print("TUNING CATBOOST NATIVE CATEGORICAL")
print("=" * 70)

cat_native_study, cat_native_params = (
    tune_catboost_native()
)

print(
    "Best CV R2:",
    cat_native_study.best_value
)

print(
    "Best parameters:",
    cat_native_params
)


X_old_cat = X_old.copy()
X_new_cat = X_new.copy()

for col in categorical_features:

    X_old_cat[col] = (
        X_old_cat[col].astype(str)
    )

    X_new_cat[col] = (
        X_new_cat[col].astype(str)
    )


cat_indices = [
    X_old_cat.columns.get_loc(col)
    for col in categorical_features
]


cat_native_model = CatBoostRegressor(
    **cat_native_params
)


print(
    "\nTraining final native CatBoost..."
)

start_time = time.time()

cat_native_model.fit(
    X_old_cat,
    y_old,
    cat_features=cat_indices,
    verbose=False
)

cat_native_training_time = (
    time.time() - start_time
)


start_time = time.time()

cat_native_prediction = (
    cat_native_model.predict(
        X_new_cat
    )
)

cat_native_prediction_time = (
    time.time() - start_time
)


# Metrics

cat_native_full = calculate_metrics(
    y_new,
    cat_native_prediction
)

cat_native_ood = calculate_subset_metrics(
    y_new,
    cat_native_prediction,
    ood_mask
)

cat_native_extreme = calculate_subset_metrics(
    y_new,
    cat_native_prediction,
    extreme_mask
)


print(
    "\nFull temporal test:"
)

print(
    f"R2   : {cat_native_full['R2']:.4f}"
)

print(
    f"RMSE : {cat_native_full['RMSE']:.4f}"
)

print(
    f"MAE  : {cat_native_full['MAE']:.4f}"
)


print(
    "\nStrict OOD:"
)

print(
    f"R2   : {cat_native_ood['R2']:.4f}"
)

print(
    f"RMSE : {cat_native_ood['RMSE']:.4f}"
)

print(
    f"MAE  : {cat_native_ood['MAE']:.4f}"
)


print(
    "\nExtreme-tail stress test:"
)

print(
    f"R2   : {cat_native_extreme['R2']:.4f}"
)

print(
    f"RMSE : {cat_native_extreme['RMSE']:.4f}"
)

print(
    f"MAE  : {cat_native_extreme['MAE']:.4f}"
)


# Save model

cat_native_model_path = os.path.join(
    MODEL_DIR,
    "ro3_catboost_native_temporal.joblib"
)

joblib.dump(
    cat_native_model,
    cat_native_model_path
)


# Save predictions

cat_native_prediction_df = new_df.copy()

cat_native_prediction_df[
    "prediction"
] = cat_native_prediction

cat_native_prediction_df[
    "residual"
] = (
    y_new.to_numpy()
    - cat_native_prediction
)

cat_native_prediction_df[
    "is_ood"
] = ood_mask

cat_native_prediction_df[
    "is_extreme_tail"
] = extreme_mask


cat_native_prediction_path = os.path.join(
    TABLE_DIR,
    "ro3_catboost_native_temporal_predictions.csv"
)

cat_native_prediction_df.to_csv(
    cat_native_prediction_path,
    index=False
)


# Add results

all_results.extend(
    [

        {
            "model":
                "CatBoost (Native categorical)",

            "subset":
                "Full temporal test",

            "R2":
                cat_native_full["R2"],

            "RMSE":
                cat_native_full["RMSE"],

            "MAE":
                cat_native_full["MAE"],

            "N":
                cat_native_full["N"],

            "training_time_s":
                cat_native_training_time,

            "prediction_time_s":
                cat_native_prediction_time
        },

        {
            "model":
                "CatBoost (Native categorical)",

            "subset":
                "Strict OOD",

            "R2":
                cat_native_ood["R2"],

            "RMSE":
                cat_native_ood["RMSE"],

            "MAE":
                cat_native_ood["MAE"],

            "N":
                cat_native_ood["N"],

            "training_time_s":
                np.nan,

            "prediction_time_s":
                np.nan
        },

        {
            "model":
                "CatBoost (Native categorical)",

            "subset":
                "Extreme-tail stress test",

            "R2":
                cat_native_extreme["R2"],

            "RMSE":
                cat_native_extreme["RMSE"],

            "MAE":
                cat_native_extreme["MAE"],

            "N":
                cat_native_extreme["N"],

            "training_time_s":
                np.nan,

            "prediction_time_s":
                np.nan
        }
    ]
)


all_predictions[
    "CatBoost (Native categorical)"
] = cat_native_prediction




TUNING CATBOOST NATIVE CATEGORICAL
Best CV R2: 0.9107185416028292
Best parameters: {'iterations': 465, 'depth': 6, 'learning_rate': 0.15843594540919131, 'l2_leaf_reg': 5.445513592141071, 'loss_function': 'RMSE', 'random_seed': 42, 'verbose': False, 'thread_count': -1}

Training final native CatBoost...

Full temporal test:
R2   : -0.3257
RMSE : 47.3919
MAE  : 39.7705

Strict OOD:
R2   : -2.7083
RMSE : 55.3895
MAE  : 49.6712

Extreme-tail stress test:
R2   : -0.9074
RMSE : 50.2302
MAE  : 43.1215


In [31]:
# ================================================================
# 30. SAVE ROBUSTNESS RESULTS
# ================================================================

results_df = pd.DataFrame(
    all_results
)


results_path = os.path.join(
    TABLE_DIR,
    "ood_robustness_supervised_models.csv"
)


results_df.to_csv(
    results_path,
    index=False
)



In [32]:
# ================================================================
# 31. SAVE COMBINED PREDICTIONS
# ================================================================

predictions_path = os.path.join(
    TABLE_DIR,
    "ro3_predictions_supervised_models.csv"
)


all_predictions.to_csv(
    predictions_path,
    index=False
)



In [33]:
# ================================================================
# 32. SAVE ROBUSTNESS THRESHOLDS
# ================================================================

thresholds_df = pd.DataFrame(
    [
        {
            "S_old_N":
                len(X_old),

            "S_new_N":
                len(X_new),

            "engine_size_min_S_old":
                engine_min,

            "engine_size_max_S_old":
                engine_max,

            "engine_size_p05_S_old":
                engine_p05,

            "engine_size_p95_S_old":
                engine_p95,

            "strict_OOD_N":
                int(ood_mask.sum()),

            "extreme_tail_N":
                int(extreme_mask.sum())
        }
    ]
)


thresholds_path = os.path.join(
    TABLE_DIR,
    "ro3_robustness_thresholds.csv"
)


thresholds_df.to_csv(
    thresholds_path,
    index=False
)



In [34]:
# ================================================================
# 33. FINAL RESULTS
# ================================================================

print("\n")
print("=" * 70)
print("RO3 TEMPORAL ROBUSTNESS EVALUATION COMPLETE")
print("=" * 70)

print(
    "\nFinal robustness results:"
)

print(
    results_df[
        [
            "model",
            "subset",
            "R2",
            "RMSE",
            "MAE",
            "N"
        ]
    ].to_string(index=False)
)


print(
    "\nSaved files:"
)

print(
    results_path
)

print(
    predictions_path
)

print(
    thresholds_path
)

print(
    "\nModels saved in:"
)

print(
    MODEL_DIR
)



RO3 TEMPORAL ROBUSTNESS EVALUATION COMPLETE

Final robustness results:
                        model                   subset        R2      RMSE       MAE    N
        Random Forest (Tuned)       Full temporal test -0.902596 56.773911 48.105959 4002
        Random Forest (Tuned)               Strict OOD -4.412618 66.918423 61.305739 2346
        Random Forest (Tuned) Extreme-tail stress test -1.789311 60.742355 52.974226 3343
              XGBoost (Tuned)       Full temporal test -0.399570 48.693702 39.424454 4002
              XGBoost (Tuned)               Strict OOD -3.002851 57.547537 50.354618 2346
              XGBoost (Tuned) Extreme-tail stress test -1.054470 52.130644 43.491867 3343
     Ridge Regression (Tuned)       Full temporal test  0.453941 30.415540 23.518507 4002
     Ridge Regression (Tuned)               Strict OOD -0.173888 31.164149 25.147658 2346
     Ridge Regression (Tuned) Extreme-tail stress test  0.246577 31.569123 24.825785 3343
             LightGBM (Tune